# radcoolpv — radiative cooling of silicon PV

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gsilvaoelker/radcoolpv-py/blob/main/docs/site/notebooks/radcoolpv_colab.ipynb)

Edit the YAML, run the cell, read the numbers. Nothing is installed on your own
machine and nothing here needs a compiled electromagnetic solver.

**Runtime → Run all** takes about a minute and produces the **temperatures** a
module settles at, the **powers** that balance there, the full set of **PV
parameters**, and the figures. No cell will stop and wait for you.

One published paper is reproduced group by group in three companion notebooks:
[A — optics](validation_a_optics.ipynb),
[B — cooling](validation_b_cooling.ipynb),
[C — the full cell](validation_c_pv.ipynb).

## Set up the runtime

Colab runtimes are temporary. Run this again after a reset.

In [ ]:
import os, subprocess, sys
from pathlib import Path
from IPython.display import Markdown, display

PROJECT = Path("/content/radcoolpv-py")
if not PROJECT.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "main",
                    "https://github.com/gsilvaoelker/radcoolpv-py.git", str(PROJECT)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--editable", "."],
               cwd=PROJECT, check=True)
os.chdir(PROJECT)

from radcoolpv import config, pipeline, report
print("radcoolpv ready in", PROJECT)

## PV parameters, with no solver

Optics, energy balance and the single-diode cell, end to end, from a committed
free-form structure.

In [ ]:
CASE = "examples/freeform_pv.yaml"
report.summary(pipeline.run(config.load_cases(CASE)[0]))

## Edit this and run your own case

Everything below is yours: the wavelength range, the layer thicknesses, the
materials, the geometry, the ambient temperature, the convection coefficient.

Three things worth knowing:

* **The wavelength range must fit inside every material's table.** The error
  names the file that is too narrow. `RII_Olmon_2012_ev_Au` stops at 24.93 µm.
* **Commenting a key out reverts it to the code default, which is often the
  opposite of what you wanted.** `# thermal: false` turns thermal *on*. Set
  values explicitly. Commenting out a list item, such as a `structure` layer,
  is safe.
* **`n` is not a free parameter.** Every spectral integral is trapezoidal on
  your grid, so the grid has to resolve the bands that matter.

In [ ]:
%%writefile my_case.yaml
# Cooling curve for a surface whose emittance you supply.
run:
  optics: false               # no solver: read the spectrum from a file
  thermal: true
  plots: true
  mode: cooling_curve
  write_outputs: true
  results_dir: results/my_case
  optics_results: validation/data/fig5a_measured_emittance.txt
  optics_results_angles: hemispherical
  optics_results_emittance_column: 3     # 1 bare, 2 flat silica, 3 cylinders

simulation:
  wavelength: {min: 2.0, max: 16.0, n: 281}
  angles: hemispherical

thermal:
  ambient_temperature: 300.0       # K
  convection_coefficient: 12.54    # W/m2-K, everything non-radiative
  absorbed_solar_power: 808.0      # W/m2 absorbed, not incident irradiance
  equilibrium: auto
  cooling_temperature: {min: 260.0, max: 380.0, n: 121}

In [ ]:
CASE = "my_case.yaml"
report.summary(pipeline.run(config.load_cases(CASE)[0]))

## Run it on your own data

Leave `MY_DATA = False` and this cell does nothing — the case above has already
run. Set it to `True` and it opens a file picker, wires your file into the same
case, and runs it. You never have to edit the YAML to use your own spectrum.

Your file needs wavelength in micrometres in the first column and emittance in
another; set `MY_COLUMN` to that column's index. If you are handing it a
spectrum radcoolpv itself exported, set `MY_COLUMN = None` instead — those
files already say which column is which.

This case runs `mode: cooling_curve`, which solves no cell, so it reports
temperatures and powers whatever you feed it. For the PV parameters as well,
switch the case to `mode: standard` and give it a spectrum reaching below about
1.1 µm — below the band gap there is nothing for the cell to convert.

In [ ]:
MY_DATA = False      # True -> pick a file and run this case on it
MY_COLUMN = 3         # emittance column; None if the file is a radcoolpv export

if MY_DATA:
    from google.colab import files
    name = next(iter(files.upload()))      # Colab saves it beside the notebook
    cfg = config.load_cases(CASE)[0]
    cfg.run.optics = False
    cfg.run.optics_results = name
    cfg.run.optics_results_emittance_column = MY_COLUMN
    report.summary(pipeline.run(cfg))
else:
    print("Ran the case above. Set MY_DATA = True to run it on your own file.")